<a href="https://colab.research.google.com/github/AjMing/BigData/blob/main/Streaming/foreachRDD_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Complete foreachRDD Tutorial

## Understanding foreachRDD in Spark Streaming

### What is foreachRDD?
`foreachRDD` is used in **Spark Streaming** to apply a function to each RDD in a stream.

### When to use it?
- Save streaming data to a database
- Write data to files
- Send data to external systems
- Perform custom processing on each batch

### Key Difference:
- `foreach()` - Works on **individual elements**
- `foreachRDD()` - Works on **entire RDDs** in a stream

**Let's learn with examples!**

---
##  Setup

In [1]:
# Install PySpark
print("Installing PySpark...")
!pip install pyspark -q
print(" Installation complete!")

Installing PySpark...
 Installation complete!


In [2]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext
from pyspark import SparkContext
from pyspark.sql.functions import *
import time
import os
import warnings

warnings.filterwarnings('ignore')
print(" Libraries imported!")

 Libraries imported!


---
##  Part 1: Regular RDD foreach() vs foreachRDD()

Let's first understand the difference with regular RDDs

In [11]:
# Create Spark Context
try:
    sc.stop()
except:
    pass

sc = SparkContext.getOrCreate()
sc.setLogLevel("ERROR")

print("="*70)
print("RDD foreach() - Operates on EACH ELEMENT")
print("="*70 + "\n")

# Example 1: foreach on regular RDD
numbers = [1, 2, 3, 4, 5]
numbers_rdd = sc.parallelize(numbers)

print("Example 1: Print each number")
print("Numbers:", numbers)
print("\nUsing foreach():")

# foreach processes each element but doesn't return anything
def print_element(x):
    print(f"  Processing: {x}")

numbers_rdd.foreach(print_element)

# Example 2: Count words in each line
print("\n" + "="*70)
print("Example 2: Process each line")
print("="*70 + "\n")

sentences = [
    "Hello world",
    "Spark is awesome",
    "RDDs are powerful"
]
sentences_rdd = sc.parallelize(sentences)

def process_line(line):
    word_count = len(line.split())
    print(f"  Line: '{line}' has {word_count} words")

sentences_rdd.foreach(process_line)

print("\n foreach() demonstrated!")

RDD foreach() - Operates on EACH ELEMENT

Example 1: Print each number
Numbers: [1, 2, 3, 4, 5]

Using foreach():

Example 2: Process each line


 foreach() demonstrated!


---
##  Part 2: foreachRDD() with Structured Streaming

Since DStreams are older, we'll use **Structured Streaming** with foreachBatch (modern equivalent)

In [4]:
# Create Spark Session for Structured Streaming
spark = SparkSession.builder \
    .appName("foreachRDD_Tutorial") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Session created!")


Spark Session created!


### Example 1: Print Each Batch (RDD) to Console

In [5]:
print("="*70)
print("Example 1: foreachBatch - Print Each RDD")
print("="*70 + "\n")

# Create streaming source
stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Add processing
processed = stream.select(
    col("timestamp"),
    col("value"),
    (col("value") % 10).alias("category")
)

# Define function to process each batch (RDD)
def process_batch(batch_df, batch_id):
    """
    This function is called for EACH batch (RDD) in the stream

    Parameters:
    - batch_df: The DataFrame for this batch
    - batch_id: Sequential ID of the batch
    """
    print(f"\n{'='*60}")
    print(f"Processing Batch ID: {batch_id}")
    print(f"{'='*60}")

    # Get count
    count = batch_df.count()
    print(f"   Rows in this batch: {count}")

    # Show first few rows
    if count > 0:
        print("\n   Sample data:")
        batch_df.show(5, truncate=False)

    # Category counts
    print("   Category distribution:")
    batch_df.groupBy("category").count().show()

# Use foreachBatch to process each RDD
query = processed.writeStream \
    .foreachBatch(process_batch) \
    .start()

print("Stream started... Processing batches...\n")

# Run for 20 seconds
time.sleep(20)

# Stop
query.stop()
print("\n Stream stopped!")

Example 1: foreachBatch - Print Each RDD

Stream started... Processing batches...


Processing Batch ID: 0
   Rows in this batch: 0
   Category distribution:
+--------+-----+
|category|count|
+--------+-----+
+--------+-----+


Processing Batch ID: 1
   Rows in this batch: 20

   Sample data:
+-----------------------+-----+--------+
|timestamp              |value|category|
+-----------------------+-----+--------+
|2026-04-16 23:21:40.836|0    |0       |
|2026-04-16 23:21:41.236|2    |2       |
|2026-04-16 23:21:41.636|4    |4       |
|2026-04-16 23:21:42.036|6    |6       |
|2026-04-16 23:21:42.436|8    |8       |
+-----------------------+-----+--------+
only showing top 5 rows
   Category distribution:
+--------+-----+
|category|count|
+--------+-----+
|       2|    2|
|       4|    2|
|       8|    2|
|       5|    2|
|       0|    2|
|       6|    2|
|       1|    2|
|       3|    2|
|       7|    2|
|       9|    2|
+--------+-----+


Processing Batch ID: 2
   Rows in this batch: 1

### Example 2: Save Each Batch to Files

In [6]:
print("="*70)
print("Example 2: foreachBatch - Save to Files")
print("="*70 + "\n")

# Create output directory
output_dir = "/tmp/streaming_output"
os.makedirs(output_dir, exist_ok=True)

# Create streaming source
stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

# Transform data
student_data = stream.select(
    (col("value") % 500).alias("student_id"),
    ((col("value") % 40) + 60).alias("score"),
    col("timestamp")
).withColumn(
    "grade",
    when(col("score") >= 90, "A")
    .when(col("score") >= 80, "B")
    .when(col("score") >= 70, "C")
    .otherwise("D")
)

# Function to save each batch
def save_batch(batch_df, batch_id):
    """
    Save each batch to a separate file
    """
    print(f"\nSaving Batch {batch_id}...")

    if batch_df.count() > 0:
        # Save as CSV
        file_path = f"{output_dir}/batch_{batch_id}.csv"

        batch_df.coalesce(1).write \
            .mode("overwrite") \
            .option("header", "true") \
            .csv(file_path)

        print(f"    Saved {batch_df.count()} rows to {file_path}")

        # Show summary
        print("   Grade distribution:")
        batch_df.groupBy("grade").count().show()
    else:
        print("  Empty batch, skipping...")

# Start streaming with foreachBatch
query = student_data.writeStream \
    .foreachBatch(save_batch) \
    .start()

print(" Saving batches to files...\n")

# Run for 15 seconds
time.sleep(15)

# Stop
query.stop()

# List saved files
print("\n📁 Files saved:")
!ls -lh /tmp/streaming_output/

print("\nAll batches saved!")

Example 2: foreachBatch - Save to Files

💾 Saving batches to files...


Saving Batch 0...
  Empty batch, skipping...

Saving Batch 1...
    Saved 10 rows to /tmp/streaming_output/batch_1.csv
   Grade distribution:
+-----+-----+
|grade|count|
+-----+-----+
|    D|   10|
+-----+-----+


Saving Batch 2...
    Saved 10 rows to /tmp/streaming_output/batch_2.csv
   Grade distribution:
+-----+-----+
|grade|count|
+-----+-----+
|    C|   10|
+-----+-----+


Saving Batch 3...
    Saved 10 rows to /tmp/streaming_output/batch_3.csv
   Grade distribution:
+-----+-----+
|grade|count|
+-----+-----+
|    B|   10|
+-----+-----+


Saving Batch 4...
    Saved 10 rows to /tmp/streaming_output/batch_4.csv
   Grade distribution:
+-----+-----+
|grade|count|
+-----+-----+
|    A|   10|
+-----+-----+


Saving Batch 5...
    Saved 10 rows to /tmp/streaming_output/batch_5.csv
   Grade distribution:
+-----+-----+
|grade|count|
+-----+-----+
|    D|   10|
+-----+-----+


Saving Batch 6...
    Saved 10 rows to /tm

### Example 3: Calculate Running Statistics with foreachBatch

In [12]:
print("="*70)
print("Example 3: Running Statistics Across Batches")
print("="*70 + "\n")

# Global variables to track statistics
total_records = 0
total_score = 0
batch_count = 0

# Create streaming source
stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 8) \
    .load()

# Generate student scores
scores_stream = stream.select(
    (col("value") % 300).alias("student_id"),
    ((col("value") % 40) + 60).alias("score")
)

# Function to track running statistics
def track_statistics(batch_df, batch_id):
    """
    Track cumulative statistics across all batches
    """
    global total_records, total_score, batch_count

    batch_count += 1

    if batch_df.count() > 0:
        # Batch statistics
        batch_records = batch_df.count()
        batch_sum = batch_df.agg({"score": "sum"}).collect()[0][0]
        batch_avg = batch_df.agg({"score": "avg"}).collect()[0][0]
        batch_max = batch_df.agg({"score": "max"}).collect()[0][0]
        batch_min = batch_df.agg({"score": "min"}).collect()[0][0]

        # Update global statistics
        total_records += batch_records
        total_score += batch_sum

        # Calculate running average
        running_avg = total_score / total_records

        # Print results
        print(f"\n{'='*60}")
        print(f"Batch {batch_id} Statistics")
        print(f"{'='*60}")
        print(f"\nTHIS BATCH:")
        print(f"   Records: {batch_records}")
        print(f"   Average Score: {batch_avg:.2f}")
        print(f"   Max Score: {batch_max}")
        print(f"   Min Score: {batch_min}")

        print(f"\n RUNNING TOTALS (All {batch_count} batches):")
        print(f"   Total Records: {total_records}")
        print(f"   Running Average: {running_avg:.2f}")
        print(f"   Total Score Sum: {total_score}")

# Start streaming
query = scores_stream.writeStream \
    .foreachBatch(track_statistics) \
    .start()

print(" Tracking statistics...\n")

# Run for 20 seconds
time.sleep(20)

# Stop
query.stop()

# Final summary
print(f"\n{'='*60}")
print(" FINAL SUMMARY")
print(f"{'='*60}")
print(f"Total Batches Processed: {batch_count}")
print(f"Total Records: {total_records}")
print(f"Overall Average Score: {total_score/total_records if total_records > 0 else 0:.2f}")
print("\n Complete!")

Example 3: Running Statistics Across Batches



Py4JJavaError: An error occurred while calling o1987.load.
: org.apache.spark.SparkException: [INTERNAL_ERROR] No active or default Spark session found SQLSTATE: XX000
	at org.apache.spark.SparkException$.internalError(SparkException.scala:92)
	at org.apache.spark.SparkException$.internalError(SparkException.scala:96)
	at org.apache.spark.sql.SparkSessionCompanion.$anonfun$active$2(SparkSession.scala:1031)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.SparkSessionCompanion.$anonfun$active$1(SparkSession.scala:1031)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.SparkSessionCompanion.active(SparkSession.scala:1030)
	at org.apache.spark.sql.execution.streaming.sources.RateStreamProvider.getTable(RateStreamProvider.scala:66)
	at org.apache.spark.sql.internal.connector.SimpleTableProvider.getOrLoadTable(SimpleTableProvider.scala:35)
	at org.apache.spark.sql.internal.connector.SimpleTableProvider.inferSchema(SimpleTableProvider.scala:41)
	at or

### Example 4: Multiple Operations per Batch

In [8]:
print("="*70)
print("Example 4: Multiple Operations per Batch")
print("="*70 + "\n")

# Create directories
high_scores_dir = "/tmp/high_scores"
low_scores_dir = "/tmp/low_scores"
os.makedirs(high_scores_dir, exist_ok=True)
os.makedirs(low_scores_dir, exist_ok=True)

# Create streaming source
stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

# Generate student data
students = stream.select(
    (col("value") % 500).alias("student_id"),
    ((col("value") % 40) + 60).alias("score"),
    col("timestamp")
)

# Function to do multiple things per batch
def multi_operation_batch(batch_df, batch_id):
    """
    Perform multiple operations on each batch:
    1. Print summary
    2. Save high scores to one location
    3. Save low scores to another
    4. Count by category
    """
    print(f"\n{'='*60}")
    print(f"⚡ Processing Batch {batch_id}")
    print(f"{'='*60}")

    if batch_df.count() == 0:
        print("   Empty batch, skipping...")
        return

    # 1. Print summary
    total = batch_df.count()
    avg_score = batch_df.agg({"score": "avg"}).collect()[0][0]
    print(f"\n Summary:")
    print(f"   Total students: {total}")
    print(f"   Average score: {avg_score:.2f}")

    # 2. Filter and save high scores (>= 85)
    high_scores = batch_df.filter(col("score") >= 85)
    high_count = high_scores.count()

    if high_count > 0:
        high_scores.coalesce(1).write \
            .mode("append") \
            .parquet(high_scores_dir)
        print(f"\n High scores (>= 85): {high_count} students saved")

    # 3. Filter and save low scores (< 70)
    low_scores = batch_df.filter(col("score") < 70)
    low_count = low_scores.count()

    if low_count > 0:
        low_scores.coalesce(1).write \
            .mode("append") \
            .parquet(low_scores_dir)
        print(f"  Low scores (< 70): {low_count} students saved")

    # 4. Categorize and count
    categorized = batch_df.withColumn(
        "category",
        when(col("score") >= 90, "Excellent")
        .when(col("score") >= 80, "Good")
        .when(col("score") >= 70, "Average")
        .otherwise("Needs Improvement")
    )

    print(f"\n📈 Score Categories:")
    categorized.groupBy("category").count() \
        .orderBy("category") \
        .show(truncate=False)

# Start streaming
query = students.writeStream \
    .foreachBatch(multi_operation_batch) \
    .start()

print(" Processing batches with multiple operations...\n")

# Run for 20 seconds
time.sleep(20)

# Stop
query.stop()

# Check saved files
print("\n" + "="*60)
print("📁 Saved Files Summary")
print("="*60)

# Read and display high scores
try:
    high_df = spark.read.parquet(high_scores_dir)
    print(f"\n Total High Scores Saved: {high_df.count()}")
    print("Sample:")
    high_df.orderBy("score", ascending=False).show(5)
except:
    print("\n No high scores saved")

# Read and display low scores
try:
    low_df = spark.read.parquet(low_scores_dir)
    print(f"\n  Total Low Scores Saved: {low_df.count()}")
    print("Sample:")
    low_df.orderBy("score").show(5)
except:
    print("\n  No low scores saved")

print("\n Complete!")

Example 4: Multiple Operations per Batch

 Processing batches with multiple operations...


⚡ Processing Batch 0
   Empty batch, skipping...

⚡ Processing Batch 1

 Summary:
   Total students: 10
   Average score: 64.50
  Low scores (< 70): 10 students saved

📈 Score Categories:
+-----------------+-----+
|category         |count|
+-----------------+-----+
|Needs Improvement|10   |
+-----------------+-----+


⚡ Processing Batch 2

 Summary:
   Total students: 20
   Average score: 79.50

 High scores (>= 85): 5 students saved

📈 Score Categories:
+--------+-----+
|category|count|
+--------+-----+
|Average |10   |
|Good    |10   |
+--------+-----+


⚡ Processing Batch 3

 Summary:
   Total students: 20
   Average score: 79.50

 High scores (>= 85): 10 students saved
  Low scores (< 70): 10 students saved

📈 Score Categories:
+-----------------+-----+
|category         |count|
+-----------------+-----+
|Excellent        |10   |
|Needs Improvement|10   |
+-----------------+-----+


⚡ Proces

ERROR:py4j.clientserver:There was an exception while executing the Python Proxy on the Python Side.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 644, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyspark/sql/utils.py", line 165, in call
    raise e
  File "/usr/local/lib/python3.12/dist-packages/pyspark/sql/utils.py", line 162, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_5975/3219392430.py", line 43, in multi_operation_batch
    avg_score = batch_df.agg({"score": "avg"}).collect()[0][0]
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyspark/sql/classic/dataframe.py", line 443, in collect
    sock_info = self._jdf.collectToPython()
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  


📁 Saved Files Summary

 Total High Scores Saved: 60
Sample:
+----------+-----+--------------------+
|student_id|score|           timestamp|
+----------+-----+--------------------+
|       119|   99|2026-04-16 23:22:...|
|        39|   99|2026-04-16 23:22:...|
|        79|   99|2026-04-16 23:22:...|
|       159|   99|2026-04-16 23:22:...|
|       118|   98|2026-04-16 23:22:...|
+----------+-----+--------------------+
only showing top 5 rows

  Total Low Scores Saved: 50
Sample:
+----------+-----+--------------------+
|student_id|score|           timestamp|
+----------+-----+--------------------+
|       120|   60|2026-04-16 23:22:...|
|         0|   60|2026-04-16 23:22:...|
|        40|   60|2026-04-16 23:22:...|
|       160|   60|2026-04-16 23:22:...|
|        80|   60|2026-04-16 23:22:...|
+----------+-----+--------------------+
only showing top 5 rows

 Complete!


### Example 5: Simulating Database Writes with foreachBatch

In [9]:
print("="*70)
print("Example 5: Simulate Database Writes")
print("="*70 + "\n")

# Simulate a simple in-memory "database"
database = {
    'enrollments': [],
    'total_students': 0,
    'courses': {}
}

# Create streaming source
stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Course list
courses = ['Math', 'Science', 'English', 'History', 'Art']

# Generate enrollment data
enrollments = stream.select(
    (col("value") % 300).alias("student_id"),
    array([lit(c) for c in courses])[
        (col("value") % len(courses)).cast("int")
    ].alias("course"),
    col("timestamp").alias("enrollment_time")
)

# Function to "write to database"
def write_to_database(batch_df, batch_id):
    """
    Simulate writing each batch to a database
    In real-world: connect to MySQL, PostgreSQL, MongoDB, etc.
    """
    global database

    print(f"\n{'='*60}")
    print(f"💾 Writing Batch {batch_id} to Database")
    print(f"{'='*60}")

    if batch_df.count() == 0:
        print("   Empty batch, skipping...")
        return

    # Collect batch data
    records = batch_df.collect()

    # Simulate INSERT into database
    for record in records:
        enrollment = {
            'student_id': record['student_id'],
            'course': record['course'],
            'enrollment_time': record['enrollment_time']
        }
        database['enrollments'].append(enrollment)

        # Update course counts
        course = record['course']
        database['courses'][course] = database['courses'].get(course, 0) + 1

    # Update total students
    database['total_students'] = len(set([e['student_id'] for e in database['enrollments']]))

    # Print database state
    print(f"\n   Inserted {len(records)} records")
    print(f"    Database Stats:")
    print(f"      Total Enrollments: {len(database['enrollments'])}")
    print(f"      Unique Students: {database['total_students']}")
    print(f"\n    Enrollments by Course:")
    for course, count in sorted(database['courses'].items(), key=lambda x: x[1], reverse=True):
        print(f"      {course}: {count}")

# Start streaming
query = enrollments.writeStream \
    .foreachBatch(write_to_database) \
    .start()

print("Simulating database writes...\n")

# Run for 20 seconds
time.sleep(20)

# Stop
query.stop()

# Final database state
print("\n" + "="*60)
print(" FINAL DATABASE STATE")
print("="*60)
print(f"\nTotal Enrollments: {len(database['enrollments'])}")
print(f"Unique Students: {database['total_students']}")
print(f"\nCourse Popularity:")
for course, count in sorted(database['courses'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {course}: {count} students")

# Show sample enrollments
print(f"\nSample Enrollments (first 5):")
for enrollment in database['enrollments'][:5]:
    print(f"  Student {enrollment['student_id']} -> {enrollment['course']}")

print("\n Database simulation complete!")

Example 5: Simulate Database Writes

Simulating database writes...


💾 Writing Batch 0 to Database
   Empty batch, skipping...

💾 Writing Batch 1 to Database

   Inserted 5 records
    Database Stats:
      Total Enrollments: 5
      Unique Students: 5

    Enrollments by Course:
      Math: 1
      English: 1
      Art: 1
      Science: 1
      History: 1

💾 Writing Batch 2 to Database

   Inserted 5 records
    Database Stats:
      Total Enrollments: 10
      Unique Students: 10

    Enrollments by Course:
      Math: 2
      English: 2
      Art: 2
      Science: 2
      History: 2

💾 Writing Batch 3 to Database

   Inserted 5 records
    Database Stats:
      Total Enrollments: 15
      Unique Students: 15

    Enrollments by Course:
      Math: 3
      English: 3
      Art: 3
      Science: 3
      History: 3

💾 Writing Batch 4 to Database

   Inserted 5 records
    Database Stats:
      Total Enrollments: 20
      Unique Students: 20

    Enrollments by Course:
      Math: 4
   

---
##  Comparison Summary

### foreach() vs foreachBatch()

| Feature | foreach() | foreachBatch() |
|---------|-----------|----------------|
| **Works on** | Individual elements | Entire batches (RDDs/DataFrames) |
| **Use case** | Print, simple operations | Save to DB, files, complex processing |
| **Context** | Regular RDDs | Streaming (DStreams/Structured) |
| **Return value** | None | None |
| **Performance** | Element-by-element | Batch processing (faster) |

### When to use foreachBatch()?

 Sving streaming data to databases  
 Writing to multiple destinations  
 Applying complex transformations per batch  
 Custom partitioning logic  
 Triggering external systems  
 Maintaining state across batches  

### Real-World Examples:

1. **E-commerce**: Save orders to database in real-time
2. **IoT**: Process sensor batches and store in time-series DB
3. **Logs**: Parse log batches and save alerts to different tables
4. **Analytics**: Calculate metrics per batch and update dashboards
5. **ETL**: Extract, transform each batch, load to data warehouse

---
##  Cleanup

In [10]:
# Stop all active streams
print("Cleaning up...\n")

for stream in spark.streams.active:
    print(f"   Stopping: {stream.name}")
    stream.stop()

print("\nAll streams stopped!")


Cleaning up...


All streams stopped!


---
##  Summary

### What You Learned:

 Difference between `foreach()` and `foreachBatch()`  
 How to process each batch in a stream  
 Saving streaming data to files  
 Calculating running statistics  
 Performing multiple operations per batch  
 Simulating database writes  

### Key Takeaways:

1. **foreachBatch()** gives you full control over each batch
2. Perfect for custom sink operations (databases, files, APIs)
3. Can perform multiple operations on same batch
4. Useful for maintaining state across batches
5. More efficient than element-by-element processing

### Practice Exercises:

1. ✏️ Modify Example 2 to save JSON instead of CSV
2. ✏️ Add error handling in foreachBatch functions
3. ✏️ Create a batch that filters and saves to 3 different locations
4. ✏️ Implement a simple alert system for low scores
5. ✏️ Build a batch processor that sends emails (simulated)

### Resources:

- [Spark Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming-programming-guide.html)
- [Structured Streaming + foreachBatch](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#using-foreach-and-foreachbatch)

---

**Great job completing this tutorial!**
